In [1]:
import os
import numpy as np
import librosa
import parselmouth
import pandas as pd

DATASET_PATH = "User_Voice"
users = sorted(os.listdir(DATASET_PATH))  # U001-U023

def extract_features(file_path, sr=16000):
    y, sr = librosa.load(file_path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    y = y / np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else y

    # Parselmouth features
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0

    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

    energy = np.mean(y**2)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0

    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)

    features = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    return features

# Save all features per user
all_features = []
all_labels = []

for user in users:
    user_path = os.path.join(DATASET_PATH, user)
    files = [f for f in os.listdir(user_path) if f.endswith(".wav")]
    for f in files:
        file_path = os.path.join(user_path, f)
        feats = extract_features(file_path)
        all_features.append(feats)
        all_labels.append(user)

X = np.array(all_features)
y = np.array(all_labels)
print("Features extracted:", X.shape)


Features extracted: (230, 26)


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import random

class TripletVoiceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.users = list(set(y))
        self.user_to_indices = {u: np.where(y==u)[0] for u in self.users}

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        anchor = self.X[idx]
        label = self.y[idx]
        pos_idx = idx
        while pos_idx == idx:
            pos_idx = np.random.choice(self.user_to_indices[label])
        positive = self.X[pos_idx]

        neg_user = random.choice([u for u in self.users if u != label])
        neg_idx = np.random.choice(self.user_to_indices[neg_user])
        negative = self.X[neg_idx]

        return torch.tensor(anchor, dtype=torch.float32), \
               torch.tensor(positive, dtype=torch.float32), \
               torch.tensor(negative, dtype=torch.float32)


In [3]:
import torch.nn as nn
import torch.optim as optim

class VoiceEmbeddingNet(nn.Module):
    def __init__(self, input_dim=26, embedding_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):
        return self.net(x)


In [4]:
from torch.nn.functional import cosine_similarity

device = 'cuda' if torch.cuda.is_available() else 'cpu'
input_dim = X.shape[1]
embedding_dim = 64

model = VoiceEmbeddingNet(input_dim=input_dim, embedding_dim=embedding_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
triplet_dataset = TripletVoiceDataset(X, y)
triplet_loader = DataLoader(triplet_dataset, batch_size=16, shuffle=True)

margin = 0.5
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for anchor, positive, negative in triplet_loader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        anchor_embed = model(anchor)
        pos_embed = model(positive)
        neg_embed = model(negative)

        pos_sim = cosine_similarity(anchor_embed, pos_embed)
        neg_sim = cosine_similarity(anchor_embed, neg_embed)

        # Triplet loss: max(0, margin + neg - pos)
        loss = torch.relu(margin + neg_sim - pos_sim).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(triplet_loader):.4f}")

torch.save(model.state_dict(), "voice_embedding_model.pth")
print("Embedding model trained and saved.")


Epoch 1/50, Loss: 0.2886
Epoch 2/50, Loss: 0.1545
Epoch 3/50, Loss: 0.1461
Epoch 4/50, Loss: 0.1258
Epoch 5/50, Loss: 0.1244
Epoch 6/50, Loss: 0.1022
Epoch 7/50, Loss: 0.1010
Epoch 8/50, Loss: 0.0921
Epoch 9/50, Loss: 0.0801
Epoch 10/50, Loss: 0.0690
Epoch 11/50, Loss: 0.0613
Epoch 12/50, Loss: 0.0741
Epoch 13/50, Loss: 0.0717
Epoch 14/50, Loss: 0.0517
Epoch 15/50, Loss: 0.0796
Epoch 16/50, Loss: 0.0472
Epoch 17/50, Loss: 0.0493
Epoch 18/50, Loss: 0.0588
Epoch 19/50, Loss: 0.0483
Epoch 20/50, Loss: 0.0442
Epoch 21/50, Loss: 0.0611
Epoch 22/50, Loss: 0.0652
Epoch 23/50, Loss: 0.0417
Epoch 24/50, Loss: 0.0497
Epoch 25/50, Loss: 0.0387
Epoch 26/50, Loss: 0.0427
Epoch 27/50, Loss: 0.0390
Epoch 28/50, Loss: 0.0278
Epoch 29/50, Loss: 0.0368
Epoch 30/50, Loss: 0.0395
Epoch 31/50, Loss: 0.0210
Epoch 32/50, Loss: 0.0346
Epoch 33/50, Loss: 0.0389
Epoch 34/50, Loss: 0.0233
Epoch 35/50, Loss: 0.0266
Epoch 36/50, Loss: 0.0326
Epoch 37/50, Loss: 0.0333
Epoch 38/50, Loss: 0.0330
Epoch 39/50, Loss: 0.

In [5]:
model.eval()
user_embeddings = {}

with torch.no_grad():
    for user in users:
        indices = np.where(y==user)[0]
        feats_user = torch.tensor(X[indices], dtype=torch.float32).to(device)
        embeds = model(feats_user).cpu().numpy()
        user_embeddings[user] = np.mean(embeds, axis=0)  # average embedding


In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

DATASET_PATH = "User_Voice"
users = sorted(os.listdir(DATASET_PATH))

all_features = []

def extract_features(file_path):
    # Use the same feature extraction function you already have
    # Make sure it returns a 1D numpy array
    import librosa, parselmouth
    y, sr = librosa.load(file_path, sr=16000, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    y = y / np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else y
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    energy = np.mean(y**2)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)
    features = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    return features

# Collect features
for user in users:
    user_path = os.path.join(DATASET_PATH, user)
    for f in os.listdir(user_path):
        if f.endswith(".wav"):
            feats = extract_features(os.path.join(user_path, f))
            all_features.append(feats)

X = np.array(all_features)
print("Feature shape:", X.shape)


Feature shape: (230, 26)


In [9]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
scaler.fit(X)  # fit on dataset features
joblib.dump(scaler, "scaler.save")
print("Scaler saved to scaler.save")


Scaler saved to scaler.save


In [10]:
import torch
import numpy as np

# Load your trained model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
input_dim = X.shape[1]
embedding_dim = 64

model = VoiceEmbeddingNet(input_dim=input_dim, embedding_dim=embedding_dim).to(device)
model.load_state_dict(torch.load("voice_embedding_model.pth", map_location=device))
model.eval()

user_embeddings = {}
with torch.no_grad():
    for user in users:
        user_features = []
        user_path = os.path.join(DATASET_PATH, user)
        for f in os.listdir(user_path):
            if f.endswith(".wav"):
                feats = extract_features(os.path.join(user_path, f))
                feats_std = scaler.transform(feats.reshape(1,-1))
                x_tensor = torch.tensor(feats_std, dtype=torch.float32).to(device)
                embed = model(x_tensor).cpu().numpy().reshape(-1)
                embed = embed / np.linalg.norm(embed)
                user_features.append(embed)
        user_embeddings[user] = np.mean(user_features, axis=0)  # average embedding per user

np.save("user_embeddings.npy", user_embeddings)
print("User embeddings saved to user_embeddings.npy")


User embeddings saved to user_embeddings.npy


In [1]:
import os
import numpy as np
import librosa
import parselmouth
import sounddevice as sd
import soundfile as sf
import torch

# ---------------- Feature Extraction ----------------
def extract_features(file_path, sr=16000):
    y, sr = librosa.load(file_path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    y = y / np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else y

    # Parselmouth features
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0

    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

    energy = np.mean(y**2)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)

    features = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    return features

# ---------------- Voice Embedding Model ----------------
class VoiceEmbeddingNet(torch.nn.Module):
    def __init__(self, input_dim=26, embedding_dim=64):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, embedding_dim)
        )

    def forward(self, x):
        return self.net(x)

# ---------------- Record Short Clip ----------------
def record_clip(duration=5, filename="temp.wav"):
    print(f"Recording {duration} seconds...")
    audio = sd.rec(int(duration*16000), samplerate=16000, channels=1)
    sd.wait()
    sf.write(filename, audio, 16000)
    return filename

# ---------------- Live Verification ----------------
def verify_live_multi_clip(model, scaler, user_embeddings, num_clips=3, duration=5, threshold=0.85, device='cpu'):
    model.eval()
    embeddings = []
    for i in range(num_clips):
        print(f"--- Clip {i+1}/{num_clips} ---")
        file = record_clip(duration=duration, filename=f"clip_{i+1}.wav")
        feats = extract_features(file)
        feats_std = scaler.transform(feats.reshape(1,-1))
        x_tensor = torch.tensor(feats_std, dtype=torch.float32).to(device)
        with torch.no_grad():
            embed = model(x_tensor).cpu().numpy().reshape(-1)
            embed = embed / np.linalg.norm(embed)  # normalize
            embeddings.append(embed)
    avg_embed = np.mean(embeddings, axis=0)
    avg_embed = avg_embed / np.linalg.norm(avg_embed)

    # Compare with stored user embeddings
    best_user = "No existing user"
    best_sim = -1
    for user, u_embed in user_embeddings.items():
        sim = np.dot(avg_embed, u_embed)
        if sim > best_sim:
            best_sim = sim
            best_user = user
    if best_sim < threshold:
        best_user = "No existing user"
    return best_user, best_sim

# ---------------- Example Usage ----------------
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Load model
    input_dim = 26  # match your feature dimension
    embedding_dim = 64
    model = VoiceEmbeddingNet(input_dim=input_dim, embedding_dim=embedding_dim).to(device)
    model.load_state_dict(torch.load("voice_embedding_model.pth", map_location=device))

    # Load dataset scaler and user embeddings
    import joblib
    scaler = joblib.load("scaler.save")  # make sure you saved the dataset scaler
    user_embeddings = np.load("user_embeddings.npy", allow_pickle=True).item()

    # Ensure embeddings are normalized
    for user in user_embeddings:
        user_embeddings[user] = user_embeddings[user] / np.linalg.norm(user_embeddings[user])

    # Verify live speaker
    speaker, similarity = verify_live_multi_clip(model, scaler, user_embeddings, num_clips=3, duration=5)
    print(f"Identified Speaker: {speaker}")
    print(f"Similarity: {similarity:.4f}")


--- Clip 1/3 ---
Recording 5 seconds...
--- Clip 2/3 ---
Recording 5 seconds...
--- Clip 3/3 ---
Recording 5 seconds...
Identified Speaker: No existing user
Similarity: 0.8500
